# Human-in-the-Loop: el humano como validador de acciones

## ¿De qué trata este notebook?

Este notebook introduce un patrón de diseño fundamental para sistemas de IA en producción: **Human-in-the-Loop (HITL)**.

### El problema que resuelve

Los agentes de IA pueden hacer cosas muy útiles: buscar información, enviar emails, borrar registros, publicar en redes sociales... Pero algunas de esas acciones son **irreversibles o de alto riesgo**.

¿Querrías que un agente borrara registros de tu base de datos sin pedirte permiso? ¿O que enviara un email en tu nombre a un cliente importante?

Human-in-the-Loop resuelve esto: **el agente se pausa antes de ejecutar acciones sensibles** y espera tu aprobación.

### La analogía

Es como un asistente muy eficiente que prepara todo para ti, pero antes de firmar un contrato o enviar una propuesta importante, te lo pone encima de la mesa y espera tu visto bueno.

```
┌──────────────────────────────────────────────────┐
│  Agente planifica acción                          │
│         ↓                                         │
│  ¿Es una acción sensible?                         │
│     Sí ↓                    No → ejecuta directo  │
│  ⏸️  PAUSA - espera aprobación humana             │
│         ↓                                         │
│  Humano: ✅ Aprobar / ❌ Rechazar                  │
│         ↓                                         │
│  Agente continúa o cancela                        │
└──────────────────────────────────────────────────┘
```

### Casos de uso típicos
- Envío de emails a clientes
- Transacciones financieras
- Borrado de datos
- Publicaciones en redes sociales
- Cambios en configuraciones de sistemas

---

## Paso 0 — Instalación y configuración

In [ ]:
# !pip install -q langchain langchain-google-genai langgraph

In [ ]:
from dotenv import load_dotenv
import os

# Cargamos la API Key de forma segura desde el archivo .env
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY_001")

# En Google Colab, usa esta línea en su lugar:
# API_KEY = userdata.get('GEMINI_API_KEY')

---

## Paso 1 — Definir las herramientas

### Herramientas seguras vs. herramientas sensibles

No todas las herramientas tienen el mismo nivel de riesgo. En este ejemplo tenemos tres:

| Herramienta | Riesgo | ¿Requiere aprobación? |
|-------------|--------|----------------------|
| `buscar_informacion` | Ninguno — solo lee | No |
| `enviar_email` | Medio — acción externa irreversible | Sí |
| `borrar_registros` | Alto — destruye datos | Sí |

### ¿Qué es `@tool`?

`@tool` es un decorador de Python que convierte una función normal en una herramienta que el agente puede usar. La descripción entre `"""` es fundamental: el agente la lee para saber cuándo y cómo usar cada herramienta.

### ¿Por qué estas son simulaciones?

En un sistema real, `enviar_email` llamaría a una API de correo (Gmail, Outlook...) y `borrar_registros` ejecutaría un DELETE en la base de datos. Aquí usamos simulaciones (`print`) para demostrar el concepto sin efectos reales.

In [ ]:
from langchain.tools import tool

# ─── Herramienta SEGURA: solo lee, no modifica nada ───────────────────────────
@tool
def buscar_informacion(query: str) -> str:
    """Busca información general. Es segura y no requiere aprobación."""
    # Simulación de una base de conocimiento
    # En un sistema real, aquí habría una llamada a un motor de búsqueda o una base de datos
    respuestas = {
        "python": "Python es un lenguaje de programación de alto nivel creado en 1991.",
        "ia": "La Inteligencia Artificial simula procesos de inteligencia humana.",
        "langchain": "LangChain es un framework para construir aplicaciones con LLMs."
    }
    # Busca si alguna palabra clave está en la consulta del usuario
    for clave, valor in respuestas.items():
        if clave in query.lower():
            return valor
    return f"Resultado de búsqueda para: {query}"


# ─── Herramienta SENSIBLE: acción externa que no se puede deshacer ─────────────
@tool
def enviar_email(destinatario: str, asunto: str, cuerpo: str) -> str:
    """Envía un email al destinatario especificado. REQUIERE APROBACIÓN HUMANA."""
    # En producción aquí iría la lógica real de envío (API de correo)
    # Aquí simulamos mostrando lo que se enviaría
    print(f"  📧 [SIMULACIÓN] Email enviado a {destinatario}")
    print(f"     Asunto: {asunto}")
    print(f"     Cuerpo: {cuerpo[:100]}...")
    return f"Email enviado correctamente a {destinatario}"


# ─── Herramienta MUY SENSIBLE: destruye datos de forma permanente ──────────────
@tool
def borrar_registros(tabla: str, condicion: str) -> str:
    """Borra registros de la base de datos. MUY SENSIBLE - requiere aprobación."""
    # En producción: DELETE FROM tabla WHERE condicion
    print(f"  🗑️ [SIMULACIÓN] Borrando de {tabla} donde {condicion}")
    return f"Registros borrados de {tabla} con condición: {condicion}"

print("✅ Herramientas definidas:")
print("   ✔️ buscar_informacion (SEGURA)")
print("   ⚠️ enviar_email (SENSIBLE)")
print("   🚨 borrar_registros (MUY SENSIBLE)")

---

## Paso 2 — Crear el agente con interrupciones

### El parámetro clave: `interrupt_before`

`interrupt_before=["tools"]` le dice al agente: **"antes de ejecutar cualquier herramienta, párate y espera"**.

Sin este parámetro, el agente ejecutaría todas las herramientas automáticamente. Con él, el agente planifica la acción pero no la ejecuta hasta que alguien (el humano) le diga que continúe.

### ¿Por qué necesitamos el checkpointer aquí?

Cuando el agente se pausa, tiene que poder **reanudar desde donde lo dejó**. El checkpointer guarda el estado en el momento de la pausa. Cuando apruebas la acción, el agente recupera ese estado y continúa.

Sin checkpointer, no hay forma de reanudar: el agente olvidaría todo lo que estaba haciendo.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# El modelo que usa el agente
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# El checkpointer es OBLIGATORIO para poder reanudar después de una pausa
checkpointer = InMemorySaver()

# Lista de herramientas que consideramos sensibles (requieren aprobación humana)
HERRAMIENTAS_SENSIBLES = ["enviar_email", "borrar_registros"]

# Creamos el agente con la interrupción activada
# interrupt_before=["tools"] → el agente se pausa ANTES de ejecutar cualquier herramienta
# Esto nos da la oportunidad de revisar qué herramienta va a usar y con qué argumentos
agente_hitl = create_agent(
    model=llm,
    tools=[buscar_informacion, enviar_email, borrar_registros],
    checkpointer=checkpointer,
    interrupt_before=["tools"]  # ← La clave de Human-in-the-Loop
)

print("✅ Agente HITL creado")
print("   El agente pausará ANTES de ejecutar herramientas para revisión humana")

---

## Paso 3 — El flujo completo: pausar, revisar, aprobar o rechazar

### ¿Qué hace la función `ejecutar_con_aprobacion`?

Esta función implementa el flujo completo de HITL:

1. **Primera ejecución** (`invoke` con el mensaje): el agente procesa el mensaje, decide qué herramienta usar, y SE PAUSA antes de ejecutarla
2. **Revisión**: mostramos al humano qué herramienta quiere usar el agente y con qué argumentos
3. **Decisión del humano**: el humano aprueba (`s`) o rechaza (`n`)
4. **Reanudación** (`invoke(None, config)`): si aprueba, el agente continúa desde donde se pausó

### ¿Qué significa `invoke(None, config)`?

En LangGraph (el motor subyacente), `invoke(None, config)` significa: **"reanuda la ejecución pausada para este thread, sin añadir nuevos mensajes"**. El `None` indica que no hay nuevo input; solo continúa.

### ¿Qué son `tool_calls`?

Cuando el agente decide que necesita usar una herramienta, genera un `AIMessage` especial que contiene `tool_calls`: una lista de las herramientas que quiere llamar y con qué argumentos. El agente NO las ha ejecutado aún — solo ha declarado su intención.

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import Command

def ejecutar_con_aprobacion(agente, mensaje: str, thread_id: str):
    """Ejecuta el agente con pausa para aprobación humana."""
    # Cada conversación tiene su propio thread_id para poder reanudarla
    config = {"configurable": {"thread_id": thread_id}}
    
    print(f"\n{'='*60}")
    print(f"📨 Solicitud: {mensaje}")
    print('='*60)
    
    # Primera ejecución: el agente planifica pero SE PAUSA antes de ejecutar herramientas
    # El agente devuelve un estado con la herramienta que quiere usar (pero sin haberla ejecutado)
    estado = agente.invoke(
        {"messages": [HumanMessage(content=mensaje)]},
        config=config
    )
    
    # Inspeccionamos el último mensaje del agente
    # Si tiene tool_calls, significa que el agente quiere usar herramientas pero está pausado
    ultimo_mensaje = estado["messages"][-1]
    
    if hasattr(ultimo_mensaje, 'tool_calls') and ultimo_mensaje.tool_calls:
        # El agente quiere usar herramientas — mostramos los detalles para revisión
        for tool_call in ultimo_mensaje.tool_calls:
            nombre_herramienta = tool_call['name']    # Nombre de la función que quiere llamar
            args = tool_call['args']                   # Argumentos con los que la quiere llamar
            
            # Verificamos si esta herramienta está en nuestra lista de sensibles
            es_sensible = nombre_herramienta in HERRAMIENTAS_SENSIBLES
            
            print(f"\n⏸️  PAUSA: El agente quiere usar '{nombre_herramienta}'")
            print(f"   Argumentos: {args}")
            
            if es_sensible:
                # Herramienta sensible → pedimos aprobación explícita al humano
                print(f"\n⚠️  Esta herramienta es SENSIBLE y requiere aprobación.")
                # En un entorno real (web, app), aquí aparecería un botón de UI
                # En este notebook, usamos input() para simular la decisión humana
                aprobacion = input("\n👤 ¿Aprobar esta acción? [s/n]: ").strip().lower()
                
                if aprobacion == 's':
                    print("✅ Aprobado. El agente continúa...")
                    # invoke(None, config) reanuda la ejecución desde donde se pausó
                    # None = no hay nuevo mensaje del usuario; solo continúa
                    estado_final = agente.invoke(None, config=config)
                    print("\n🤖 Respuesta final:")
                    print(estado_final["messages"][-1].content)
                else:
                    print("❌ Rechazado. Cancelando operación.")
                    print("\n🤖 Respuesta: La operación fue cancelada por el usuario.")
            else:
                # Herramienta segura → no necesitamos aprobación, continuamos automáticamente
                print("✅ Herramienta segura, continuando automáticamente...")
                estado_final = agente.invoke(None, config=config)
                print("\n🤖 Respuesta final:")
                print(estado_final["messages"][-1].content)
    else:
        # El agente respondió directamente sin necesitar herramientas
        print("\n🤖 Respuesta directa (sin herramientas):")
        print(ultimo_mensaje.content)

print("✅ Función de aprobación definida")

In [ ]:
# Prueba 1: búsqueda de información
# buscar_informacion es SEGURA → el agente la ejecutará sin pedir aprobación
ejecutar_con_aprobacion(
    agente_hitl,
    "Busca información sobre Python",
    thread_id="test-busqueda-001"  # Cada prueba tiene su propio thread para no mezclar estados
)

In [ ]:
# Prueba 2: envío de email
# enviar_email es SENSIBLE → el agente se pausará y pedirá tu aprobación
# Cuando te pregunte, escribe 's' para aprobar o 'n' para rechazar
ejecutar_con_aprobacion(
    agente_hitl,
    "Envía un email a profesor@master.es con asunto 'Entrega práctica' y cuerpo 'Adjunto mi práctica del máster'",
    thread_id="test-email-001"
)

---

## Paso 4 — Lógica granular de aprobación

### ¿Por qué necesitamos lógica más granular?

No toda ejecución de una herramienta sensible tiene el mismo riesgo. Por ejemplo:
- Enviar un email a `test@test.local` (entorno de pruebas) → riesgo bajo, aprobar automáticamente
- Enviar un email a `director@empresa.com` (entorno real) → riesgo alto, pedir aprobación

La función `revisar_herramienta` implementa esta lógica: examina no solo el nombre de la herramienta sino también sus argumentos para decidir si necesita aprobación.

### La regla de negocio

| Herramienta | Condición | ¿Aprobación? |
|-------------|-----------|-------------|
| `buscar_informacion` | Siempre | No |
| `enviar_email` | Destinatario es `@test.local` | No |
| `enviar_email` | Destinatario es cualquier otro | Sí |
| `borrar_registros` | Siempre | Sí |

In [ ]:
def revisar_herramienta(nombre: str, args: dict) -> bool:
    """Decide si una herramienta necesita aprobación humana."""
    # Lista de herramientas que SIEMPRE requieren aprobación, sin excepciones
    herramientas_siempre_aprobar = ["borrar_registros"]
    
    # Herramientas que requieren aprobación SOLO en ciertos contextos
    herramientas_aprobar_si_produccion = ["enviar_email"]
    
    # ¿La herramienta está en la lista de siempre aprobar?
    if nombre in herramientas_siempre_aprobar:
        return True  # Siempre pedir aprobación
    
    # Para enviar_email: depende del destinatario
    if nombre in herramientas_aprobar_si_produccion:
        destinatario = args.get("destinatario", "")
        # Si el destinatario termina en @test.local → es un entorno de pruebas → no hace falta aprobación
        # Si es cualquier otro → es real → sí hace falta aprobación
        return not destinatario.endswith("@test.local")
    
    return False  # Por defecto, las herramientas no requieren aprobación


# Demostramos la lógica con casos concretos
print("🔍 Ejemplos de decisiones de aprobación:")
casos = [
    ("buscar_informacion", {"query": "python"}),                              # Segura → no
    ("enviar_email", {"destinatario": "test@test.local"}),                   # Email a test → no
    ("enviar_email", {"destinatario": "director@empresa.com"}),              # Email real → sí
    ("borrar_registros", {"tabla": "usuarios", "condicion": "id=1"}),        # Siempre → sí
]

for nombre, args in casos:
    # Llamamos a nuestra función para saber si necesita aprobación
    necesita = revisar_herramienta(nombre, args)
    icono = "⚠️ REQUIERE APROBACIÓN" if necesita else "✅ Auto-aprobado"
    print(f"  {nombre}({args}): {icono}")

---

## Resumen y puntos clave

| Concepto | Descripción | Por qué importa |
|----------|-------------|----------------|
| **`interrupt_before=["tools"]`** | Pausa el agente antes de ejecutar herramientas | Permite revisión humana antes de actuar |
| **`checkpointer`** | Guarda el estado en el momento de la pausa | Sin él, no se puede reanudar |
| **`thread_id`** | Identifica la conversación pausada | Permite reanudar la conversación correcta |
| **`invoke(None, config)`** | Reanuda la ejecución pausada | El `None` indica que no hay nuevo input |
| **`tool_calls`** | Herramientas que el agente quiere ejecutar | Aquí revisamos qué acción va a tomar |

## Ejercicio propuesto

1. Crea una herramienta `publicar_en_rrss(plataforma, mensaje)` que simule publicar en redes sociales
2. Añádela como herramienta sensible al agente
3. Implementa en `revisar_herramienta` una regla: pide aprobación solo si el mensaje contiene las palabras "urgente", "oferta" o "descuento"
4. Bonus: añade un log de todas las aprobaciones y rechazos con `datetime.now()` para saber cuándo ocurrieron